# NER Dataset Collection (Colab)

This notebook collects and standardizes datasets for your domain adaptation NER project:

- CoNLL-2003
- WNUT-17
- SciERC

## Download strategy

To avoid Hugging Face access issues, this notebook downloads from public URLs:

- CoNLL-2003 split files from public GitHub raw links
- WNUT-17 split files from public GitHub raw links
- SciERC processed archive from the official UW SciIE URL

## Data flow (`raw` -> `processed` -> Google Drive)

1. Download original files into `data/raw`
2. Parse/convert into a common sentence-level JSONL format in `data/processed`
3. Build a single report file (`collection_report.json`)
4. Copy the whole `data/` folder to Google Drive for persistence across Colab sessions

## Output schema (processed JSONL)

Each row contains:

- `id`
- `tokens`
- `tags` (BIO format)
- `source_dataset`
- `split`

Run cells top-to-bottom.

In [1]:
!pip -q install "datasets==2.19.2" pandas pyarrow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
import json
import tarfile
import urllib.request
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path('/content/ner_capstone')
RAW = BASE / 'data' / 'raw'
PROCESSED = BASE / 'data' / 'processed'

for p in [RAW, PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

print('Workspace:', BASE)

Workspace: /content/ner_capstone


In [3]:
def save_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + '\n')


def summarize_and_save(dataset_name, split_rows):
    out_dir = PROCESSED / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    summary = {'dataset': dataset_name, 'splits': {}, 'label_counts': {}}

    for split, rows in split_rows.items():
        save_jsonl(out_dir / f'{dataset_name}_{split}.jsonl', rows)

        label_counter = Counter()
        token_count = 0
        for r in rows:
            token_count += len(r['tokens'])
            label_counter.update(r['tags'])

        summary['splits'][split] = {
            'sentences': len(rows),
            'tokens': token_count,
        }
        summary['label_counts'][split] = dict(label_counter.most_common())

    with open(out_dir / 'summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=True)

    print(f'[OK] {dataset_name} ->', out_dir)


def download_file(url, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, out_path)
    print(f'[OK] downloaded {url} -> {out_path}')


def read_conll_like_file(path):
    rows = []
    tokens, tags = [], []

    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if tokens:
                    rows.append((tokens, tags))
                    tokens, tags = [], []
                continue

            if line.startswith('-DOCSTART-'):
                continue

            parts = line.split()
            token = parts[0]
            tag = parts[-1]
            tokens.append(token)
            tags.append(tag)

    if tokens:
        rows.append((tokens, tags))

    return rows


def convert_conll_rows_to_jsonl(rows, dataset_name, split_name):
    out = []
    for i, (tokens, tags) in enumerate(rows):
        out.append({
            'id': f'{split_name}-{i}',
            'tokens': tokens,
            'tags': tags,
            'source_dataset': dataset_name,
            'split': split_name,
        })
    return out


URLS = {
    'conll2003': {
        'train': 'https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train',
        'validation': 'https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa',
        'test': 'https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb',
    },
    'wnut17': {
        'train': 'https://raw.githubusercontent.com/leondz/emerging_entities_17/master/wnut17train.conll',
        'validation': 'https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.dev.conll',
        'test': 'https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.test.annotated',
    },
}

## Step A: Download and process CoNLL-2003 + WNUT-17

These next cells:

- fetch split files (`train`, `validation`, `test`) from public GitHub URLs
- save untouched text files under `data/raw/<dataset>/`
- parse CoNLL-style lines into sentence-level records
- write standardized JSONL files to `data/processed/<dataset>/`

This keeps both source files and cleaned files for reproducibility.

In [4]:
conll_raw_dir = RAW / 'conll2003'
conll_raw_dir.mkdir(parents=True, exist_ok=True)

conll_split_rows = defaultdict(list)
for split, url in URLS['conll2003'].items():
    out_fp = conll_raw_dir / f'{split}.txt'
    download_file(url, out_fp)
    rows = read_conll_like_file(out_fp)
    conll_split_rows[split] = convert_conll_rows_to_jsonl(rows, 'conll2003', split)

summarize_and_save('conll2003', conll_split_rows)

[OK] downloaded https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train -> /content/ner_capstone/data/raw/conll2003/train.txt
[OK] downloaded https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa -> /content/ner_capstone/data/raw/conll2003/validation.txt
[OK] downloaded https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb -> /content/ner_capstone/data/raw/conll2003/test.txt
[OK] conll2003 -> /content/ner_capstone/data/processed/conll2003


In [5]:
wnut_raw_dir = RAW / 'wnut17'
wnut_raw_dir.mkdir(parents=True, exist_ok=True)

wnut_split_rows = defaultdict(list)
for split, url in URLS['wnut17'].items():
    out_fp = wnut_raw_dir / f'{split}.txt'
    download_file(url, out_fp)
    rows = read_conll_like_file(out_fp)
    wnut_split_rows[split] = convert_conll_rows_to_jsonl(rows, 'wnut17', split)

summarize_and_save('wnut17', wnut_split_rows)

[OK] downloaded https://raw.githubusercontent.com/leondz/emerging_entities_17/master/wnut17train.conll -> /content/ner_capstone/data/raw/wnut17/train.txt
[OK] downloaded https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.dev.conll -> /content/ner_capstone/data/raw/wnut17/validation.txt
[OK] downloaded https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.test.annotated -> /content/ner_capstone/data/raw/wnut17/test.txt
[OK] wnut17 -> /content/ner_capstone/data/processed/wnut17


## Step B: Download and process SciERC

SciERC is downloaded as a tarball into `data/raw/scierc_processed.tar.gz`, then extracted under `data/raw/scierc_processed/`.

We then:

- read the official JSON split files (`train/dev/test`)
- convert entity spans to token-level BIO tags
- export standardized JSONL files into `data/processed/scierc/`

In [6]:
SCIERC_URL = 'http://nlp.cs.washington.edu/sciIE/data/sciERC_processed.tar.gz'
scierc_tar = RAW / 'scierc_processed.tar.gz'

urllib.request.urlretrieve(SCIERC_URL, scierc_tar)
print('[OK] Downloaded:', scierc_tar)

scierc_extract = RAW / 'scierc_processed'
scierc_extract.mkdir(parents=True, exist_ok=True)

with tarfile.open(scierc_tar, 'r:gz') as tar:
    tar.extractall(path=scierc_extract)

print('[OK] Extracted to:', scierc_extract)

[OK] Downloaded: /content/ner_capstone/data/raw/scierc_processed.tar.gz


/tmp/ipykernel_1817/1079293004.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=scierc_extract)


[OK] Extracted to: /content/ner_capstone/data/raw/scierc_processed


In [7]:
def scierc_doc_to_sent_rows(doc, split_name, doc_id):
    sentences = doc['sentences']
    ner_by_sent = doc.get('ner', [[] for _ in sentences])

    rows = []
    abs_cursor = 0

    for si, tokens in enumerate(sentences):
        tags = ['O'] * len(tokens)
        sent_ners = ner_by_sent[si] if si < len(ner_by_sent) else []

        for span in sent_ners:
            if len(span) < 3:
                continue
            s, e, label = span[0], span[1], span[2]

            if abs_cursor <= s <= e < abs_cursor + len(tokens):
                rs = s - abs_cursor
                re = e - abs_cursor
            elif 0 <= s <= e < len(tokens):
                rs = s
                re = e
            else:
                continue

            tags[rs] = f'B-{label}'
            for k in range(rs + 1, re + 1):
                tags[k] = f'I-{label}'

        rows.append({
            'id': f'{doc_id}-sent-{si}',
            'tokens': tokens,
            'tags': tags,
            'source_dataset': 'scierc',
            'split': split_name,
        })

        abs_cursor += len(tokens)

    return rows


candidates = list(scierc_extract.rglob('*.json'))
print('Found JSON files:', [str(p) for p in candidates[:10]])

split_map = {}
for p in candidates:
    name = p.name.lower()
    if 'train' in name:
        split_map['train'] = p
    elif 'dev' in name or 'valid' in name or 'val' in name:
        split_map['validation'] = p
    elif 'test' in name:
        split_map['test'] = p

print('Using split files:', {k: str(v) for k, v in split_map.items()})

scierc_rows = defaultdict(list)
for split, fp in split_map.items():
    with open(fp, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
            doc = json.loads(line)
            scierc_rows[split].extend(
                scierc_doc_to_sent_rows(doc, split, doc_id=f'{split}-doc-{i}')
            )

summarize_and_save('scierc', scierc_rows)

Found JSON files: ['/content/ner_capstone/data/raw/scierc_processed/processed_data/json/dev.json', '/content/ner_capstone/data/raw/scierc_processed/processed_data/json/test.json', '/content/ner_capstone/data/raw/scierc_processed/processed_data/json/train.json']
Using split files: {'validation': '/content/ner_capstone/data/raw/scierc_processed/processed_data/json/dev.json', 'test': '/content/ner_capstone/data/raw/scierc_processed/processed_data/json/test.json', 'train': '/content/ner_capstone/data/raw/scierc_processed/processed_data/json/train.json'}
[OK] scierc -> /content/ner_capstone/data/processed/scierc


In [8]:
report = {'datasets': {}}

for ds_name in ['conll2003', 'wnut17', 'scierc']:
    summary_path = PROCESSED / ds_name / 'summary.json'
    with open(summary_path, 'r', encoding='utf-8') as f:
        report['datasets'][ds_name] = json.load(f)

report_path = PROCESSED / 'collection_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=True)

print('[OK] report:', report_path)

[OK] report: /content/ner_capstone/data/processed/collection_report.json


In [9]:
import shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

dst_data = Path('/content/drive/MyDrive/AAI590/data')
dst_data.parent.mkdir(parents=True, exist_ok=True)

if dst_data.exists():
    shutil.rmtree(dst_data)

shutil.copytree(BASE / 'data', dst_data)
print('Saved data folder to:', dst_data)

Mounted at /content/drive
Saved data folder to: /content/drive/MyDrive/AAI590/data


## Step C: Persist to Google Drive

After all datasets are converted:

- `data/raw/` contains original downloaded source files (plus extracted SciERC archive)
- `data/processed/` contains standardized JSONL files and `collection_report.json`

The next cell mounts Google Drive and copies the **entire `data/` folder** into:

- `/content/drive/MyDrive/AAI590/data`

This is the recommended persistence path so future Colab sessions can load files directly from Drive.

In [10]:
# Optional sanity check: show saved output files in Drive
for p in sorted((Path('/content/drive/MyDrive/AAI590/data/processed')).glob('*')):
    print(p)

/content/drive/MyDrive/AAI590/data/processed/collection_report.json
/content/drive/MyDrive/AAI590/data/processed/conll2003
/content/drive/MyDrive/AAI590/data/processed/scierc
/content/drive/MyDrive/AAI590/data/processed/wnut17
